In [1]:
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents.base import Document
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers.weaviate_hybrid_search import WeaviateHybridSearchRetriever
from langchain_community.embeddings.openai import OpenAIEmbeddings
import weaviate
from weaviate.client import WeaviateClient
from weaviate.util import get_valid_uuid
from dotenv import load_dotenv
from uuid import uuid4
import os
from chainlit import logger

C:\Users\Admin\AppData\Local\Temp\ipykernel_16592\2932280130.py:7: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from langchain_community.retrievers.weaviate_hybrid_search import WeaviateHybridSearchRetriever


2024-12-17 19:25:11 - Loaded .env file


In [2]:
load_dotenv()
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
WEAVIATE_API_KEY = os.getenv('WEAVIATE_API_KEY')
WEAVIATE_CLUSTER_ENV = os.getenv('WEAVIATE_CLUSTER_ENV')
print(WEAVIATE_CLUSTER_ENV)
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN

https://grpc-asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud


In [3]:
hf_llm = HuggingFacePipeline.from_model_id(model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0", task="text-generation", 
                                           model_kwargs={'max_length':512})

c:\Users\Admin\.conda\envs\text2sql\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
prompt = PromptTemplate.from_template(template = """\
  <|user|>
  Given the context, generate an SQL query for the following question. Please generate only 'SELECT' sql query and don't provide any other extra information.
  context:{context}
  question:{question}</s>
  <|assistant|>
  """)

In [8]:
prompt.format(context="CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR",
              question="Show the names of schools with a total budget between 10 to 100")

"  <|user|>\n  Given the context, generate an SQL query for the following question. Please generate only 'SELECT' sql query and don't provide any other extra information.\n  context:CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR\n  question:Show the names of schools with a total budget between 10 to 100</s>\n  <|assistant|>\n  "

In [12]:
chain = (prompt | hf_llm | StrOutputParser())

In [7]:
print(chain.invoke({'context':"CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR",
                'question':"Show the names of schools with a total budget between 10 to 100"}))

  <|user|>
  Given the context, generate an SQL query for the following question. Please generate only 'SELECT' sql query and don't provide any other extra information.
  context:CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR
  question:Show the names of schools with a total budget between 10 to 100</s>
  <|assistant|>
  
  SELECT school_name
  FROM budget
  WHERE budgeted BETWEEN 10 AND 100;


In [13]:
res = chain.invoke({'context':"CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR",
                'question':"Show the names of schools with a total budget between 10 to 100"})

In [14]:
res

"  <|user|>\n  Given the context, generate an SQL query for the following question. Please generate only 'SELECT' sql query and don't provide any other extra information.\n  context:CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR\n  question:Show the names of schools with a total budget between 10 to 100</s>\n  <|assistant|>\n  \n  SELECT school_name\n  FROM budget\n  WHERE budgeted BETWEEN 10 AND 100;"

In [10]:
type(hf_llm), type(chain)

(langchain_huggingface.llms.huggingface_pipeline.HuggingFacePipeline,
 langchain_core.runnables.base.RunnableSequence)

In [15]:
res1 = """<|user|>
  Given the context, generate an SQL query for the following question. Please generate only 'SELECT' sql query and don't provide any other extra information.
  context:CREATE TABLE endowment (school_id VARCHAR, amount INTEGER); CREATE TABLE budget (school_id VARCHAR, budgeted INTEGER); CREATE TABLE school (school_name VARCHAR, school_id VARCHAR
  question:Show the names of schools with a total budget between 10 to 100</s>
  <|assistant|>
  
  SELECT school_name
  FROM budget
  WHERE budgeted BETWEEN 10 AND 100;"""

In [17]:
res1.split("<|assistant|>")[1].strip()

'SELECT school_name\n  FROM budget\n  WHERE budgeted BETWEEN 10 AND 100;'

VectorDatabase Weaviate

In [3]:
loader = CSVLoader("Data/temp12.csv")
doc = loader.load()
doc[1].metadata

{'source': 'Data/temp12.csv', 'row': 1}

In [4]:
text_splitter = RecursiveCharacterTextSplitter(separators=["/Separate"], keep_separator=False,chunk_size=1000, chunk_overlap=200)
doc1 = text_splitter.split_documents(doc)

In [5]:
doc1[:3]
#len(doc1)

[Document(metadata={'source': 'Data/temp12.csv', 'row': 0}, page_content='question: What is the scheduled start and end date of the CR CRQ000000017039. Please note that the datatype of all Date/Time columns are BIGINT in database and the Date/Time stored in these columns will be in epoch timestamp. Please convert this epcoh timestamp to Date/Time while providing the final result.\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(scheduled_start_date BIGINT, scheduled_end_date BIGINT, infrastructure_change_id VARCHAR)\nBatch:'),
 Document(metadata={'source': 'Data/temp12.csv', 'row': 1}, page_content='question: What is the scheduled start and end date of the CR CRQ000000017059\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(scheduled_start_date BIGINT, cheduled_end_date BIGINT, infrastructure_change_id VARCHAR)\nBatch:'),
 Document(metadata={'source': 'Data/temp12.csv', 'row': 2}, page_content='question: What is the Actual start and end date of the CR CRQ000000017040. Please note that t

In [7]:
'''client = weaviate.Client(url=WEAVIATE_CLUSTER_ENV, auth_client_secret= weaviate.auth.AuthApiKey(WEAVIATE_API_KEY), 
                         additional_headers={"X-OpenAI-Api-Key": OPENAI_API_KEY},)'''
client = weaviate.connect_to_weaviate_cloud(cluster_url="https://asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud", auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_API_KEY),
                                            headers={"X-OpenAI-Api-Key": OPENAI_API_KEY})

2024-12-17 19:26:55 - HTTP Request: GET https://asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud/v1/meta "HTTP/1.1 200 OK"
2024-12-17 19:26:55 - HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"


c:\Users\Admin\.conda\envs\text2sql\lib\site-packages\weaviate\warnings.py:133: DeprecationWarning: Dep005: You are using weaviate-client version 4.6.5. The latest version is 4.10.2.
            Consider upgrading to the latest version. See https://weaviate.io/developers/weaviate/client-libraries/python for details.
  warnings.warn(


In [8]:
client.is_ready()
type(client)

2024-12-17 19:27:05 - HTTP Request: GET https://asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud/v1/.well-known/ready "HTTP/1.1 200 OK"


weaviate.client.WeaviateClient

In [11]:
my_collection = client.collections.get("T2SQL")
my_collection.config.get()

2024-12-17 19:32:51 - HTTP Request: GET https://asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud/v1/schema/T2SQL "HTTP/1.1 200 OK"


_CollectionConfig(name='T2SQL', description=None, generative_config=None, inverted_index_config=_InvertedIndexConfig(bm25=_BM25Config(b=0.75, k1=1.2), cleanup_interval_seconds=60, index_null_state=False, index_property_length=False, index_timestamps=False, stopwords=_StopwordsConfig(preset=<StopwordsPreset.EN: 'en'>, additions=None, removals=None)), multi_tenancy_config=_MultiTenancyConfig(enabled=False, auto_tenant_creation=False, auto_tenant_activation=False), properties=[_Property(name='context', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_searchable=True, nested_properties=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=None, vectorizer='none')], references=[], replication_config=_ReplicationConfig(factor=1), reranker_config=None, sharding_config=_ShardingConfig(virtual_per_physical=128, desired_count=1, actual_count=1, desired_virtual_count=128, actual_virtual_count=128, key='_id', strategy='hash', function='murmur3'), vector

In [115]:
retriever = WeaviateHybridSearchRetriever(client=client, 
                                          index_name="T2SQL", 
                                          alpha=0.5, #param to balance b/w keyword and semantic search
                                          text_key="context", 
                                          attributes=[], 
                                          create_schema_if_missing=True)
type(retriever)

langchain_community.retrievers.weaviate_hybrid_search.WeaviateHybridSearchRetriever

In [12]:
def add_documents(client: WeaviateClient, index_name, text_key, documents: list, **kwargs):
    collection = client.collections.get(index_name)
    with collection.batch.dynamic() as batch:
        id = []
        for i, doc  in enumerate(documents):
            data_row = {text_key: doc.page_content}
            if "uuids" in kwargs:
                _id = kwargs["uuids"][i]
            else:
                _id = get_valid_uuid(uuid4())
            batch.add_object(properties=data_row, uuid=_id)
            id.append(_id)
    return id


In [13]:
ids = add_documents(client=client, index_name="T2SQL", text_key="context", documents=doc)

2024-12-17 19:33:28 - HTTP Request: GET https://asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud/v1/schema/T2SQL "HTTP/1.1 200 OK"
2024-12-17 19:33:28 - HTTP Request: GET https://asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud/v1/nodes "HTTP/1.1 200 OK"
2024-12-17 19:33:29 - HTTP Request: GET https://asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud/v1/nodes "HTTP/1.1 200 OK"
2024-12-17 19:33:30 - HTTP Request: GET https://asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud/v1/nodes "HTTP/1.1 200 OK"
2024-12-17 19:33:31 - HTTP Request: GET https://asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud/v1/nodes "HTTP/1.1 200 OK"
2024-12-17 19:33:32 - HTTP Request: GET https://asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud/v1/nodes "HTTP/1.1 200 OK"
2024-12-17 19:33:33 - HTTP Request: GET https://asgmftru2po7klpzl4a.c0.asia-southeast1.gcp.weaviate.cloud/v1/nodes "HTTP/1.1 200 OK"
2024-12-17 19:33:34 - HTTP Request: GET https://asgmftru2po7kl

In [14]:
collection = client.collections.get("T2SQL")
response = collection.query.hybrid(query="When was the CRQ000000017391 created?", alpha=0.25, limit=5)

In [15]:
for o in response.objects:
    print(o.properties)

{'context': 'question: What is the status of the CR CRQ000000017391\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(cr_status VARCHAR, infrastructure_change_id VARCHAR)\nBatch: '}
{'context': 'question: When was the CRQ000000019594 created\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(submit_date BIGINT, infrastructure_change_id VARCHAR)\nBatch: /Separate'}
{'context': 'question: When was the CRQ000000019284 created\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(submit_date BIGINT, infrastructure_change_id VARCHAR)\nBatch: '}
{'context': 'question: When was the CRQ000000019284 created\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(submit_date BIGINT, infrastructure_change_id VARCHAR)\nBatch: '}
{'context': 'question: When was the CRQ000000019384 created\ncontext: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(submit_date BIGINT, infrastructure_change_id VARCHAR)\nBatch: '}


In [102]:
response.objects[0].properties.keys()

dict_keys(['context', 'source', 'question', 'row'])

In [16]:
response.objects[0].properties['context'].split("\n")[1]

'context: CREATE TABLE CHG_INFRASTRUCTURE_CHANGE(cr_status VARCHAR, infrastructure_change_id VARCHAR)'

In [17]:
ids

['ed6ce1cc-e4cf-4b00-87f4-b4f6ec04d11f',
 'a93bfe18-84ff-4922-9e45-30d7b37e900a',
 '59f316ae-818f-4cce-b1b5-7042c6de007a',
 'e116ab6e-15af-4863-ab13-e63b93281eec',
 'cd11a482-496d-467f-b12a-885f84b8c36a',
 '3b7f0c46-0969-413a-a2f4-25937dfc1183',
 'c6b5f81e-e666-4a9e-9a55-2c91bb38a50d',
 '3c4c172a-29bd-4391-8087-c10845aefebb',
 'c09d7f8c-a1d2-4d47-9e1a-7985a11c45a1',
 '2ca12360-73af-47dd-ba7f-9e3367ac5443',
 '6aae947a-0860-4d3c-89ce-db44070add28',
 '45ec83b3-4774-48eb-8573-1e5d91da52fb',
 'a8ccec25-f634-40a3-81d5-bdc1ff14d35d',
 '42e6bce4-67c4-4553-91ba-7eab2a140e96',
 '1a6fcae9-d3d6-43eb-b805-246c1af3488a',
 'e03a0a07-5f82-4242-b091-3cfd3b29380d',
 'd6fa13a0-e247-473e-8576-928e87f9310c',
 '7796cf5e-04f6-4e13-927f-c408d4bbeda2',
 'b896ac45-bf0a-4774-aa63-165a7cec15d5',
 '8f6e7843-bf32-49a8-9a59-7628e86384fc',
 'f04c3de8-1727-4c11-a35e-1d2850a461fe',
 '8221ab83-cc4e-4601-8517-a015de92f2e7',
 '464ab60f-0f03-41ef-94f0-dd4b6b5813ef',
 '1fc29b7f-b004-4a45-a4f0-e04eab58bbf0',
 '3f08cee7-1ef9-